In [ ]:
import os
import torch
from src.models.vit_sae import PatchSAE, SpInViT1, SpInViT2, ViTSAE

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def load_vivit_sae_model(
    model_file, 
    model_file2=None,
    model_file3=None, 
    model_file4=None,
    model_file5=None,
    num_classes=102
):
    """
    Load a pre-trained ViViT model from the given file path.
    Args:
        args (Namespace): Arguments containing the model's state dictionary path.
    Returns:
        dict: Loaded state dictionary.
    """
    state_dict = torch.load(model_file,map_location=device)
    model = SpInViT1(num_classes=num_classes, output_attentions=True, topk=True, factor=10).to(device)
    model.load_state_dict(state_dict)
    if model_file2:
        state_dict2 = torch.load(model_file2, map_location=device)
        #model2 = SpInViT(num_classes=num_classes, output_attentions=True, topk=True, factor=10).to(device)
        model2 = ViTSAE(num_classes=num_classes, output_attentions=True).to(device)
        model2.load_state_dict(state_dict2)
    if model_file3:
        state_dict3 = torch.load(model_file3, map_location=device)
        model3 = ViTSAE(num_classes=num_classes, output_attentions=True).to(device)
        model3.load_state_dict(state_dict3)
    if model_file4:
        state_dict4 = torch.load(model_file4, map_location=device)
        model4 = SpInViT2(
            num_classes=num_classes, 
            output_attentions=True, 
            topk=True, 
            separate=True,
            relative_path='../'
        ).to(device)
        model4.load_state_dict(state_dict4)
    if model_file5:
        state_dict5 = torch.load(model_file5, map_location=device)
        model5 = SpInViT1(
            num_classes=num_classes, 
            output_attentions=True, 
            topk=False, 
            separate=False,
            relative_path='../',
            d_name="flowers102"
        ).to(device)
        model5.load_state_dict(state_dict5)

    return model#, model2, model3, model4, model5

In [ ]:
from sam2.sam2_image_predictor import SAM2ImagePredictor

dataset_dir = os.path.join("..", "flowers102") # Place any dataset here
txt_file = os.path.join("..", "record-vitsae", dataset_dir.split(os.sep)[1], "top_k_neurons.txt")
#model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_model_ce_kl_gtsrb.pth")
model_path = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_topk_model.pth")
model_path2 = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_unfrozen_model.pth")
model_path3 = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_frozen_model.pth")
model_path4 = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "separate.pth")
model_path5 = os.path.join("..", "model-vitsae",dataset_dir.split(os.sep)[1], "best_model.pth")

predictor = SAM2ImagePredictor.from_pretrained("facebook/sam2.1-hiera-large")
model, model2, model3, model4, model5 = load_vivit_sae_model(
    model_path, 
    model_file2=model_path2, 
    model_file3=model_path3, 
    model_file4=model_path4,
    model_file5=model_path5, 
    num_classes=102 # Put number of classes
)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F


def vit_sae_forward(model, image_tensor, threshold=0.2):
    """
    Perform a forward pass through the ViViT-SAE model in inference mode.
    Args:
        video_tensor (torch.Tensor): Preprocessed video input tensor.
    Returns:
        tuple: (sae_latent, att_features) as CPU tensors.
    """
    model.eval()
    with torch.no_grad():
        image_tensor = image_tensor.to(device)
        #att_weights = model.vit(image_tensor).attentions[-2][:,:,0,1:].mean(dim=1).squeeze(0)
        #att_features = model.vit(image_tensor).last_hidden_state[:, 1:] 
        _, sparse_logits, _, _ = model(image_tensor)  # (196, D) 
        #sparse_features = (sparse_features > threshold).float()
    return _, sparse_logits, _, _


def get_patch_scores(model, patch_features, att_maps, neuron_idx):
    """
    Compute per-patch contribution scores for a given SAE neuron on a single image.
    Returns a heatmap (14x14) and the original image.
    """
    model.eval()
    with torch.no_grad():
        neuron_weight = model.encoder.weight[neuron_idx]  # (D,)
        patch_scores = torch.einsum("id,d->i", patch_features.squeeze(0), neuron_weight).squeeze(0)  # (196,)
        weighted_patches = patch_scores * att_maps
        return weighted_patches.view(14, 14).cpu()


def apply_scalar_to_patches(image, patch_size=16, scalars=1.0):
    image = image[0]
    H, W, C = image.shape
    new_image = np.zeros_like(image)
    
    h_patches = H // patch_size
    w_patches = W // patch_size
    
    # Handle either a single scalar or one per patch
    for i in range(h_patches):
        for j in range(w_patches):
            patch = image[i*patch_size:(i+1)*patch_size,
                          j*patch_size:(j+1)*patch_size, :]
            scaled_patch = patch * scalars
            new_image[i*patch_size:(i+1)*patch_size,
                      j*patch_size:(j+1)*patch_size, :] = scaled_patch
    
    return new_image


def visualize_img_heatmap(image, patches_heatmap, title=None):
    """
    Overlay a heatmap on the image showing patch contributions, and show actual video frame if provided.
    """
    patches_heatmap = patches_heatmap.unsqueeze(0).unsqueeze(0)
    upsampled = F.interpolate(patches_heatmap, size=(image.shape[1], image.shape[2]), mode='bilinear').squeeze().numpy()
    image_np = image.permute(1, 2, 0).numpy()
    image_np = np.clip(image_np, 0, 1)

    plt.figure(figsize=(4, 4))
    plt.imshow(image_np)
    plt.imshow(upsampled, cmap='jet', alpha=0.5)
    plt.axis('off')
    if title:
        plt.title(title)
    plt.show()

In [ ]:
import matplotlib

def mask_patches(img, patch=16, ratio=0.1):
    C, H, W = img.shape
    ph, pw = H // patch, W // patch

    # 1) choose random patch indices
    num_mask = int(ph * pw * ratio)
    mask_idx = torch.randperm(ph * pw)[:num_mask]

    # 2) build mask grid and upsample to pixel mask
    mask = torch.zeros(ph * pw, dtype=torch.bool)
    mask[mask_idx] = True
    mask = mask.reshape(ph, pw).repeat_interleave(patch, 0).repeat_interleave(patch, 1)

    # 3) apply mask (set masked patches to 0)
    return img * (~mask)


def highlight_random_patches(img, patch=16, ratio=0.1, highlight_color=(1.0, 1.0, 0.3), alpha=0.6):
    """
    img: (3, H, W)
    highlight_color: RGB highlight
    alpha: blending strength
    """

    device = img.device
    C, H, W = img.shape
    ph, pw = H // patch, W // patch

    # 1) choose random patch indices
    num_mask = int(ph * pw * ratio)
    mask_idx = torch.randperm(ph * pw, device=device)[:num_mask]

    # 2) create patch mask (ph*pw → H×W)
    patch_mask = torch.zeros(ph * pw, device=device)
    patch_mask[mask_idx] = 1.0
    patch_mask = patch_mask.reshape(ph, pw)
    patch_mask = patch_mask.repeat_interleave(patch, 0).repeat_interleave(patch, 1)  # (H, W)

    # 3) build highlight tensor
    highlight = torch.tensor(highlight_color, device=device).view(3,1,1).expand_as(img)

    # 4) blend highlight ONLY where mask == 1
    mask3 = patch_mask.unsqueeze(0)  # (1, H, W)
    highlighted = img + mask3 * (highlight - img) * alpha

    return highlighted


def overlay_masks(image, masks):
    image = image.convert("RGBA")
    masks = 255 * masks.cpu().numpy().astype(np.uint8)
    
    n_masks = masks.shape[0]
    cmap = matplotlib.colormaps.get_cmap("rainbow").resampled(n_masks)
    colors = [
        tuple(int(c * 255) for c in cmap(i)[:3])
        for i in range(n_masks)
    ]

    for mask, color in zip(masks, colors):
        mask = Image.fromarray(mask)
        overlay = Image.new("RGBA", image.size, color + (0,))
        alpha = mask.point(lambda v: int(v * 0.5))
        overlay.putalpha(alpha)
        image = Image.alpha_composite(image, overlay)
    return image


In [ ]:
from transformers import ViTImageProcessor
from imageio.v2 import imread
from PIL import Image
import random
from matplotlib.colors import ListedColormap

# This is to only get 6 frames per video
act_tensors = []
samples = float("inf") #float("inf") # Make float("inf") if you don't want to sample and get every frame
patch_size = 16
process = ViTImageProcessor.from_pretrained("google/vit-base-patch16-224", use_fast=True)
pale_yellow ="#fff7b2"   # very light yellow
cmap = ListedColormap(["white", pale_yellow])

from datasets import load_dataset

data_name = "flowers102"
os.makedirs("../interpretability", exist_ok=True)
os.makedirs(f"../interpretability/{data_name}", exist_ok=True)
data = load_dataset(
        #"HuggingFaceM4/Caltech-101",
        #"with_background_category"
        #"ethz/food101",
        #"timm/eurosat-rgb"
        #"timm/oxford-iiit-pet"
        #"tanganke/dtd"
        "pufanyi/flowers102"
)#.shuffle(seed=1)

#import random

mod2 = True
with open(txt_file, 'r') as file:
    for i, line in enumerate(file):
        # Get neuron and file path
        neuron_idx, img_file = line.strip().split(':')
        #if not os.path.exists(img_file):
        #    continue
        #img_file = os.path.join("..", img_file.strip())
        #img_file = os.path.join("../data/flowers-102/jpg", f"{img_file.strip()[:-4]}.jpg")

        """
        if ".ppm" in img_file:
            img_file = img_file[3:-4]

        image = Image.open(f"../{img_file}")

        #pillow_image = Image.fromarray(image)
        #"""
        #"""
        # Huggingface data
        image = data['test'][int(img_file)]['image']
        #image = data['validation'][int(img_file)]['image']
        #"""
        try:
            image_tensor = process(images=image, return_tensors='pt').pixel_values
        except Exception:
            continue

        _, sparse_logits, _, _ = vit_sae_forward(model, image_tensor)
        if mod2:
            _, sparse_logits2, _, _ = vit_sae_forward(model2, image_tensor)
            neuron_activations2 = sparse_logits2.cpu()[0, :, int(neuron_idx)]
            activations2 = (neuron_activations2 > 0.5).float()

            _, sparse_logits3, _, _ = vit_sae_forward(model3, image_tensor)
            neuron_activations3 = sparse_logits3.cpu()[0, :, int(neuron_idx)]
            activations3 = (neuron_activations3 > 0.5).float()

            _, sparse_logits4, _, _ = vit_sae_forward(model4, image_tensor)
            neuron_activations4 = sparse_logits4.cpu()[0, :, int(neuron_idx)]
            activations4 = (neuron_activations4 > 0.5).float()

            _, sparse_logits5, _, _ = vit_sae_forward(model5, image_tensor)
            neuron_activations5 = sparse_logits5.cpu()[0, :, int(neuron_idx)]
            activations5 = (neuron_activations5 > 0.5).float()

        neuron_activations = sparse_logits.cpu()[0, :, int(neuron_idx)]
        #neuron_activations = torch.where(neuron_activations < 0.01, torch.tensor(0.0, device=neuron_activations.device), neuron_activations)
        activations = (neuron_activations > 0.5).float()
        
        count = (activations > 0.0).sum().numpy()
        if torch.all(activations == 0) or count < (196 * 0.0):
            #print(f"All values in neuron_activations are zero for neuron index {neuron_idx}. Skipping...")
            continue
        act_tensors.append(sparse_logits.cpu().tolist())

        #print(neuron_activations, neuron_idx)

        if int(neuron_idx) == 156:
            break
        #break

        # Get patches for image: shape -> (1, 3*16*16, 196)
        patches = F.unfold(image_tensor, kernel_size=patch_size, stride=patch_size)

        # Reshape to (196, 3, 16, 16)
        patches = patches.squeeze(0).transpose(0, 1).reshape(196, 3, patch_size, patch_size)
        
        # shape: (196, 3, 16, 16) * (196, 1, 1, 1)
        highlight_color = torch.tensor([1.0, 1.0, 0.3], device=patches.device).view(1,3,1,1)
        highlighted_patches = patches + activations[:, None, None, None] * (highlight_color - patches) * 0.7  
        if mod2:
            highlighted_patches2 = patches + activations2[:, None, None, None] * (highlight_color - patches) * 0.7
            highlighted_patches2 = highlighted_patches2.reshape(196, -1).transpose(0, 1).unsqueeze(0)   
            highlighted_image2 = F.fold(highlighted_patches2, output_size=(224, 224), kernel_size=patch_size, stride=patch_size)
            img_np2 = highlighted_image2.squeeze().permute(1, 2, 0).detach().numpy()
            img_np2 = img_np2.clip(0, 1)

            highlighted_patches3 = patches + activations3[:, None, None, None] * (highlight_color - patches) * 0.7
            highlighted_patches3 = highlighted_patches3.reshape(196, -1).transpose(0, 1).unsqueeze(0)   
            highlighted_image3 = F.fold(highlighted_patches3, output_size=(224, 224), kernel_size=patch_size, stride=patch_size)
            img_np3 = highlighted_image3.squeeze().permute(1, 2, 0).detach().numpy()
            img_np3 = img_np3.clip(0, 1)

            highlighted_patches4 = patches + activations4[:, None, None, None] * (highlight_color - patches) * 0.7
            highlighted_patches4 = highlighted_patches4.reshape(196, -1).transpose(0, 1).unsqueeze(0)   
            highlighted_image4 = F.fold(highlighted_patches4, output_size=(224, 224), kernel_size=patch_size, stride=patch_size)
            img_np4 = highlighted_image4.squeeze().permute(1, 2, 0).detach().numpy()
            img_np4 = img_np4.clip(0, 1)

            highlighted_patches5 = patches + activations5[:, None, None, None] * (highlight_color - patches) * 0.7
            highlighted_patches5 = highlighted_patches5.reshape(196, -1).transpose(0, 1).unsqueeze(0)   
            highlighted_image5 = F.fold(highlighted_patches5, output_size=(224, 224), kernel_size=patch_size, stride=patch_size)
            img_np5 = highlighted_image5.squeeze().permute(1, 2, 0).detach().numpy()
            img_np5 = img_np5.clip(0, 1)

        # reshape back to unfold shape: (1, 3*16*16, 196)
        highlighted_patches = highlighted_patches.reshape(196, -1).transpose(0, 1).unsqueeze(0)

        # fold to (1, 3, 224, 224)
        highlighted_image = F.fold(highlighted_patches, output_size=(224, 224), kernel_size=patch_size, stride=patch_size)
        

        # Permute image to (1, 224, 224, 3) and view image
        img_np = highlighted_image.squeeze().permute(1, 2, 0).detach().numpy()
        img_np = img_np.clip(0, 1)

        orig_img_np = image_tensor.squeeze().permute(1, 2, 0).detach().numpy()
        orig_img_np = orig_img_np.clip(0, 1) 


        fig, axes = plt.subplots(1, 5, figsize=(15, 5))

        # Original image
        images = [
            (img_np, "Proposed"),
            (img_np2, "Unfrozen"),
            (img_np3, "Frozen"),
            (img_np4, "Separate"),
            (img_np5, "Proposed (Not TopK)")
        ]

        # Shuffle the images
        random.shuffle(images)

        # Display them
        for ax, (img, title) in zip(axes, images):
            ax.set_title(title)
            ax.imshow(img, cmap=cmap)
            ax.axis("off")


        with torch.inference_mode(), torch.autocast("cuda", dtype=torch.bfloat16):
            predictor.set_image(image)
            masks, _, _ = predictor.predict("Outline any flowers only")
        flower_mask = np.any(masks, axis=0).astype(np.float32)
        ax.set_title("SAM2")
        ax.imshow(flower_mask, cmap=cmap)
        ax.axis("off")

        
        plt.tight_layout()
        #plt.show()

        print(f"{i} neuron")
        plt.savefig(f"../interpretability/{data_name}/nueron_{int(neuron_idx)}_{i}.png", dpi=300, bbox_inches="tight")
        plt.close()

        if i > samples: 
            break

In [ ]:
import pandas as pd
import requests
import json

result_dir = "../foldse_res"
os.makedirs(result_dir, exist_ok=True)
def train_foldse(data_name: str):
    # Enter File Paths
    path = f"../foldse/{data_name}_train.csv" # Enter the train file path
    test_path = f"../foldse/{data_name}_test.csv" # Uncomment this line if you wish to provide a test dataset

    df = pd.read_csv(path) 
    for c in df.columns:
        df[c] = df[c].astype(str)

    test_df = pd.read_csv(test_path) # Uncomment this line if you wish to provide a test dataset
    for c in test_df.columns:
        test_df[c] = test_df[c].astype(str)

    data_frame_json = df.to_json(orient='split')
    try:
        test_data_frame_json = test_df.to_json(orient='split')
    except:
        test_data_frame_json = ''

    # Set Model Parameters
    numattrs = ''#",".join(df.columns[:-1])#"" # Enter the names of numerical features seperated ',' (For eg - 'feature1,feature2,feature3')
    strattrs = ",".join(df.columns[:-1]) # Enter the names of categorical features seperated by ',' (For eg - 'feature1,feature2,feature3')
    hyp1 = "0.8" # Enter the Train Test split (Ignore if providing a seperate test dataset. If you keep hyp1 as blank or enter a value <0.5 or >=1 only rules will be generated, no testing will be done)
    hyp2 = "1.0"#"0.5" # Enter the Level of exceptions ratio
    hyp3 = "0.0005"#"0.005" # Enter the Tail ratio
    positive_value = "" # Enter the Classification label (Optional and only for binary classification. Keep unchanged otherwise)
    target_column = "label" # Enter the name of the target column
    save_model = ""#"local" # If you want to save the model ("local" will return the json model, else the model won't be saved)

    # Enter Your Username and Password
    username = "<username>" # Enter your registered email id
    password = "<password>" # Enter your password
    # If your username and password combination does not work, please try to register again with the same email id. Please contact us if that does not work.

    payload = {
        'username': username,
        'password': password,
        'data_frame_json': data_frame_json,
        'numattrs': numattrs,
        'strattrs': strattrs,
        'hyp1': hyp1,
        'hyp2': hyp2,
        'hyp3': hyp3,
        'positive_value': positive_value,
        'test_data_frame_json': test_data_frame_json,
        'label_value': target_column,
        'save_model' : save_model
    }

    response = requests.post("http://ec2-52-0-60-249.compute-1.amazonaws.com/auth/foldmodel_multicategory/", json=payload) # Uncomment if you want to run multi-category classification model.

    try:
        response_obj = response.json()
        try:
            if(response_obj['error']==None):
                # If there is no error the response_obj dictionary contains the following keys - rules, accuracy, f1_score, precision, recall, 
                # n_rules (No of rules), n_preds (No of unique predicates), size (Ruleset size), test_results (List of test dataset results only if test dataset provided).
                # Rules can be accessed with response_obj['rules'] and are returned as a string with two rules seperated by a line break (\n).

                with open(f"{result_dir}/{data_name}/full_res_vspin.json", "w") as file:
                    json.dump(response_obj, file)


                with open(f"{result_dir}/{data_name}/rules_vspin.txt", "w") as file:
                    file.write(response_obj['rules'])
                file.close()

                # Similarly you can use response_obj['accuracy'], response_obj['f1_score'], response_obj['precision'], response_obj['recall'] (These will not be generated if test dataset not provided and (hyp1>=1 or hyp1<0.5 or hyp1 is blank), 
                # response_obj['n_rules'], response_obj['n_preds'], response_obj['size'], response_obj['test_results'] (Only id test dataset provided).
                
                # If save_model = "local" - You can get the model as a json object with response_obj['model_json']. You can save the model to a file using 
                #with open('file_name.json', 'w') as f: f.write(response_obj['model_json'] + '\n')
            else:
                print('Error: ',response_obj['error'])
        except:
            print("There was an error processing your request.")
    except:
        print(response)

In [ ]:
new_dir = "../foldse"
os.makedirs(new_dir, exist_ok=True)
def fetch_data(iteration: int, data_name: str):
    train_dataloader, val_dataloader, _, num_classes, _ = get_dataloader(
        inference=False, 
        iteration=iteration,
        train_batch=1,
        val_batch=1
    )

    model_path = os.path.join(
        "..", 
        "model-vitsae",
        data_name, 
        "best_model.pth")
    model =  load_vivit_sae_model(
        model_file=model_path, 
    #     model_file2=model_path2, 
    #     model_file3=model_path3, 
    #     model_file4=model_path4,
    #     model_file5=model_path5, 
        num_classes=num_classes
    )
    model.eval()
    with torch.no_grad():
        full_data = []
        for (img, label) in train_dataloader:
            img = img.to('cuda')
            vit_features, sparse_logits, recon_logit, cls_logits = model(img)
            #feat = torch.mean(sparse_logits, dim=1).squeeze(0)
            feat = sparse_logits[:,0].squeeze(0)
            feat = (feat >= 0.5).to(torch.int).detach().cpu()
            row = torch.cat([feat, label.detach().cpu().to(torch.int)], dim=0).tolist()
            full_data.append(row)

        cols = [f"feature_{i+1}" for i in range(feat.size(0))]
        cols.append("label")

        df = pd.DataFrame(full_data, columns=cols)
        for c in cols:
            df[c] = df[c].astype(str)
        df.to_csv(f"{new_dir}/{data_name}_train.csv", index=False)


        full_data = []
        for (img, label) in val_dataloader:
            img, label = img.to('cuda'), label.to('cuda')
            vit_features, sparse_logits, recon_logit, cls_logits = model(img)
            #feat = torch.mean(sparse_logits, dim=1).squeeze(0)
            feat = sparse_logits[:,0].squeeze(0)
            feat = (feat >= 0.5).to(torch.int).detach().cpu()
            _, predicted = torch.max(cls_logits, 1)
            row = torch.cat([feat, predicted.detach().cpu().to(torch.int)], dim=0).tolist()
            full_data.append(row)
    
    
        df = pd.DataFrame(full_data, columns=cols)
        for c in cols:
            df[c] = df[c].astype(str)
        df.to_csv(f"{new_dir}/{data_name}_test.csv", index=False)

In [ ]:
import pandas as pd
from src.utils.train import get_dataloader

datasets = [
    "flowers102",
    "cars",
    "fgvc",
    "caltech101",
    "foods101",
    #"suns397",
    "dtd",
    "eurosat",
    "pets"
]

for i, data_name in enumerate(datasets):
    print(i+1, data_name)
    fetch_data(i, data_name)
    train_foldse(data_name)